*PIPELINE COMPLETO DE CAPA SILVER CON ESQUEMA ESTRICTO Y CUARENTENA*

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, trim, lower, coalesce, lit, to_date, try_to_date, when

# STEP 1: DEFINICIÓN DEL ESQUEMA ESTRICTO (Todo como StringType para la ingesta)
esquema_origen = StructType([
    StructField("id_empleado", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("departamento", StringType(), True),
    StructField("salario", StringType(), True),
    StructField("fecha_registro", StringType(), True)
])

# STEP 2: LECTURA DESDE EL VOLUMEN CON RUTA DE CUARENTENA
# (Spark mandará a la carpeta 'cuarentena' cualquier fila con errores estructurales graves)
df_bronze = spark.read \
    .option("header", "true") \
    .schema(esquema_origen) \
    .option("badRecordsPath", "/Volumes/workspace/default/csvfiles/cuarentena") \
    .csv("/Volumes/workspace/default/csvfiles/employees.csv")

In [0]:
# STEP 3: TRANSFORMACIONES Y REGLAS DE CALIDAD (Corregido y blindado contra errores de fecha)
df_silver = df_bronze \
    .dropDuplicates(["id_empleado"]) \
    .filter(col("id_empleado").isNotNull()) \
    .withColumn("nombre", trim(col("nombre"))) \
    .withColumn("departamento", when(lower(col("departamento")) == "null", lit("sin asignar"))
                           .otherwise(coalesce(lower(trim(col("departamento"))), lit("sin asignar")))) \
    .withColumn("salario", when(col("salario").cast("double") < 0, lit(0.0))
                           .otherwise(coalesce(col("salario").cast("double"), lit(0.0)))) \
    .withColumn("fecha_ingreso", coalesce(
        try_to_date(col("fecha_registro"), "yyyy-MM-dd"),  # Intenta formato estándar con guiones
        try_to_date(col("fecha_registro"), "dd/MM/yyyy")   # Si falla, intenta formato con diagonales de forma segura
    )) \
    .drop("fecha_registro")

# STEP 4: ACCIÓN FINAL PARA PROCESAR Y MOSTRAR LOS DATOS LIMPIOS

#df_silver.show()

# ==============================================================================
# STEP 4: ALOJAR EL DATAFRAME PROCESADO EN FORMATO DELTA PARQUET (CAPA SILVER)
# ==============================================================================

# Definimos la ruta de destino dentro de tu volumen seguro de Unity Catalog
ruta_destino_silver = "/Volumes/workspace/default/csvfiles/employees_silver_delta"

# Escribimos los datos en formato delta sobreescribiendo si ya existe
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_destino_silver)

#print("¡Éxito! El DataFrame df_silver ha sido alojado de forma segura en formato Delta Parquet.")

#%sql
#SELECT * FROM delta.`/Volumes/workspace/default/csvfiles/empleados_silver_delta`

In [0]:
# ==============================================================================
# PROCESAMIENTO DÍA 2: LECTURA INCREMENTAL Y APPEND EN DELTA
# ==============================================================================

# 1. LEER EL ARCHIVO DEL DÍA 2
df_bronze_dia2 = spark.read \
    .option("header", "true") \
    .schema(esquema_origen) \
    .option("badRecordsPath", "/Volumes/workspace/default/csvfiles/cuarentena") \
    .csv("/Volumes/workspace/default/csvfiles/employees_day2.csv") # <-- Apuntamos al archivo nuevo

# 2. APLICAR LAS MISMAS REGLAS DE LIMPIEZA SILVER
df_silver_dia2 = df_bronze_dia2 \
    .dropDuplicates(["id_empleado"]) \
    .filter(col("id_empleado").isNotNull()) \
    .withColumn("nombre", trim(col("nombre"))) \
    .withColumn("departamento", when(lower(col("departamento")) == "null", lit("sin asignar"))
                           .otherwise(coalesce(lower(trim(col("departamento"))), lit("sin asignar")))) \
    .withColumn("salario", when(col("salario").cast("double") < 0, lit(0.0))
                           .otherwise(coalesce(col("salario").cast("double"), lit(0.0)))) \
    .withColumn("fecha_ingreso", coalesce(
        try_to_date(col("fecha_registro"), "yyyy-MM-dd"),
        try_to_date(col("fecha_registro"), "dd/MM/yyyy")
    )) \
    .drop("fecha_registro")

# 3. ESCRIBIR EN MODO APPEND (Añadir a la carpeta Delta Parquet que ya existía)
df_silver_dia2.write \
    .format("delta") \
    .mode("append") \
    .save("/Volumes/workspace/default/csvfiles/employees_silver_delta")

print("¡Día 2 procesado e insertado correctamente en formato Delta Parquet!")


In [0]:
# 1. Leer el archivo Delta Parquet desde la ruta del volumen
df_resultado_delta = spark.read \
    .format("delta") \
    .load("/Volumes/workspace/default/csvfiles/employees_silver_delta")

# 2. Imprimir el resultado en pantalla
df_resultado_delta.show()

In [0]:
from pyspark.sql.functions import col

# 1. Leemos la tabla acumulada que tiene los duplicados
df_acumulado = spark.read.format("delta").load("/Volumes/workspace/default/csvfiles/employees_silver_delta")

# 2. VALIDACIÓN: Ordenamos por fecha (más reciente primero) y eliminamos duplicados por ID
df_gold_unico = df_acumulado \
    .sort(col("fecha_ingreso").desc()) \
    .dropDuplicates(["id_empleado"])

# 3. Imprimimos el resultado final limpio para el reporte
df_gold_unico.show()